# Algorithmic Fairness: class notebook
**HON-2501-3 · Responsible AI** · Sessions 1 and 2

<a href="https://colab.research.google.com/github/acintron-cc/hon-2501-03/blob/main/notebooks/fairness_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You don't need to write any code. Each section matches an exercise on your handout.

**How to use this notebook**
1. **File → Save a copy in Drive**, so your changes are kept.
2. **Run the cells one at a time, from top to bottom.** Click the ▶ button at the left of a cell, wait until its result appears under it (the ▶ stops spinning), then go to the next cell. The first time, Colab may warn that the notebook was not written by Google: click **Run anyway**.
3. Start with **Section 1** and wait until it prints **Ready**. Then, in **Section 2**, choose your dataset from the dropdown and click ▶.
4. When a section asks you to type something (your weights, a cutoff), do it and press the ▶ button on that cell.

> **Gemini rule:** ask Gemini to *explain* a cell or a number. Never ask it to rewrite or fix a cell. If something breaks, choose **Runtime → Restart session**, then run the cells again in order, starting at Section 1.

| Section | Handout exercise | Session |
|---|---|---|
| 1 · Setup | | 1 |
| 2 · Choose your dataset | Exercise 1 | 1 |
| 3 · Metrics by group | Exercises 1–3 | 1 |
| 4 · Reweighing | Exercise 4 | 2 |
| 5 · Threshold Optimizer | Exercise 5 | 2 |
| 6 · Compare the three models | Exercise 6 | 2 |

## Section 1 · Setup
Click the ▶ button on the cell below. It installs Fairlearn (the fairness library we use) and loads a few helper tools. It can take 1 to 2 minutes the first time; wait until it prints **Ready**.

In [ ]:
# @title Install and load the tools (just run this)
import importlib.util
if importlib.util.find_spec("fairlearn") is None:
    print("Installing Fairlearn: this takes 1 to 2 minutes the first time. Please wait...")
    !pip install -q fairlearn==0.13.0
else:
    print("Fairlearn is already installed.")

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import fairlearn
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate, count
from fairlearn.postprocessing import ThresholdOptimizer

DATA_URL = "https://raw.githubusercontent.com/acintron-cc/hon-2501-03/main/datasets/"
COLORS = ["#0F8BA6", "#E0622E", "#6A4FB3"]

def r3(x):
    """Round half up to 3 decimals, the way Excel does."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return x
    return math.floor(x * 1000 + 0.5 + 1e-9) / 1000

def gap(mf):
    # Gap = larger minus smaller value of the two groups compared in the handout.
    bg = mf.by_group.loc[COMPARE]
    return bg.max() - bg.min()

def show(table):
    """Display a table rounded the way Excel rounds."""
    display(table.map(lambda v: r3(v) if isinstance(v, (float, np.floating)) else v))

METRICS = {
    "selection rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
    "accuracy": accuracy_score,
    "n": count,
}

print("Ready. Fairlearn version:", fairlearn.__version__)

## Section 2 · Choose your dataset
Run this only after Section 1 has printed **Ready**. Pick your dataset from the dropdown on the right of the cell, then click ▶. Then run each cell below in order.

| Partner | Dataset | Group column | Groups to compare | label = 1 means |
|---|---|---|---|---|
| A | compas | race_group | African-American, Caucasian (ignore the Other row) | rearrested within 2 years (**harmful**) |
| B | adult | sex | Female, Male | income over $50K (favorable) |

In [ ]:
# @title Choose your dataset
DATASET = "compas"  # @param ["compas", "adult"]

import numpy as np    # also loaded in Section 1; repeated here so this cell works on its own
import pandas as pd
DATA_URL = "https://raw.githubusercontent.com/acintron-cc/hon-2501-03/main/datasets/"
if "fairlearn" not in globals():
    print("Note: Section 1 has not run yet. Scroll up, click the ▶ button on Section 1, wait for Ready, then run this cell again.")

CONFIG = {
    "adult": dict(target="class", group="sex", fixed="Male", second="race", compare=["Female", "Male"],
                  drop=["fnlwgt", "education", "sex", "race"],
                  meaning="income over $50K", favorable=True),
    "compas": dict(target="two_year_recid", group="race_group", fixed="African-American", second="sex",
                   compare=["African-American", "Caucasian"],
                   drop=["race", "sex", "c_charge_desc"],
                   meaning="rearrested within two years", favorable=False),
}
cfg = CONFIG[DATASET]
GROUP = cfg["group"]
COMPARE = cfg["compare"]   # the two groups compared in the handout; gaps use only these

df = pd.read_csv(DATA_URL + DATASET + "_classroom.csv")

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
groups = sorted(df[GROUP].unique())

print(f"Dataset: {DATASET}   rows: {len(df):,}   train: {len(train):,}   test: {len(test):,}")
print(f"label = 1 means: {cfg['meaning']}" + ("" if cfg["favorable"] else "   <-- a HARMFUL outcome for the person"))
print(f"Protected attribute (group column): {GROUP}   groups: {', '.join(groups)}")
print(f"Gaps compare: {COMPARE[0]} and {COMPARE[1]}" + ("   (the Other row is shown but not used in gaps)" if len(groups) > 2 else ""))
df.head()

## Section 3 · Metrics by group  *(Session 1, Exercises 1–3)*
### 3.1 · Exercise 1: group sizes and base rates (all rows)
The base rate is P(label = 1 | group): how often each group has label = 1 before any model is involved.

In [ ]:
# @title Exercise 1: base rates (all rows)
ex1 = pd.crosstab(df[GROUP], df["label"], margins=True, margins_name="Total")
ex1.columns = ["label = 0", "label = 1", "n"]
ex1["base rate"] = ex1["label = 1"] / ex1["n"]
show(ex1)

### 3.2 · Exercise 2: one confusion matrix per group (test rows)
Rows are what really happened (label); columns are what the model predicted. Compare with your spreadsheet.

In [ ]:
# @title Exercise 2: confusion matrices (test rows, original model)
for g in groups:
    t = test[test[GROUP] == g]
    cm = pd.crosstab(t["label"], t["predicted"]).reindex(index=[1, 0], columns=[1, 0], fill_value=0)
    cm.index = ["label = 1", "label = 0"]
    cm.columns = ["predicted = 1", "predicted = 0"]
    tp, fn, fp, tn = cm.iloc[0, 0], cm.iloc[0, 1], cm.iloc[1, 0], cm.iloc[1, 1]
    print(f"\n{g}  ({len(t)} test rows)   TP={tp}  FN={fn}  FP={fp}  TN={tn}")
    display(cm)

### 3.3 · Exercise 3: rates with Fairlearn's `MetricFrame` (test rows)
`MetricFrame` computes each metric separately for each group. This is the Fairlearn call to ask Gemini about.

In [ ]:
# @title Exercise 3: MetricFrame
mf = MetricFrame(
    metrics=METRICS,
    y_true=test["label"],
    y_pred=test["predicted"],
    sensitive_features=test[GROUP],
)
print("By group:")
show(mf.by_group)
print("\nGap (larger - smaller of the two compared groups):")
show(gap(mf).drop("n").to_frame("gap").T)

sr = mf.by_group.loc[COMPARE, "selection rate"]
ratio = sr.min() / sr.max()
print(f"\nSelection-rate ratio (smaller / larger): {r3(ratio):.3f}  ->",
      "passes the 80% rule" if ratio >= 0.8 else "fails the 80% rule")
if not cfg["favorable"]:
    print("Remember: in COMPAS, being 'selected' means being flagged as likely to be rearrested.")

In [ ]:
# @title Chart: rates by group
ax = mf.by_group[["selection rate", "TPR", "FPR", "accuracy"]].T.plot.bar(
    color=COLORS[:len(groups)], figsize=(8, 4), rot=0, width=0.75)
ax.set_ylim(0, 1); ax.set_ylabel("rate"); ax.set_title(f"{DATASET}: original model, test rows")
ax.legend(title=GROUP); ax.grid(axis="y", alpha=0.3); plt.show()

### 3.4 · Optional: a second attribute, or two at once
COMPAS: `sex`. Adult: `race`.
Choose an option and press ▶. Very small groups give noisy numbers, so watch the `n` column.

In [ ]:
# @title Compare by another attribute
COMPARE_BY = "second attribute"  # @param ["second attribute", "both (intersection)"]
if cfg["second"] is None:
    print("This dataset has no second attribute.")
else:
    if COMPARE_BY == "second attribute":
        sf = test[cfg["second"]]
    else:
        sf = test[[GROUP, cfg["second"]]]
    mf2 = MetricFrame(metrics=METRICS, y_true=test["label"], y_pred=test["predicted"], sensitive_features=sf)
    show(mf2.by_group)

## Section 4 · Reweighing  *(Session 2, Exercise 4)*
Reweighing gives each **train** row a weight: **weight = P(outcome) ÷ P(outcome | group)**.
### 4.1 · Step 1: the train rows' two-way table
Check this against your handout's Step 1.

In [ ]:
# @title Train rows: group x label counts
ex4 = pd.crosstab(train[GROUP], train["label"], margins=True, margins_name="Total")
ex4.columns = ["label = 0", "label = 1", "Total"]
display(ex4)

### 4.2 · YOUR WEIGHTS HERE
Find **your dataset** below and replace each `None` with the weight from your handout (3 decimals is fine).
Leave the other datasets alone. Then press ▶ on this cell and the next ones.

In [ ]:
# YOUR WEIGHTS HERE: (group, label): weight
YOUR_WEIGHTS = {
    "adult": {
        ("Female", 1): None,
        ("Female", 0): None,
        ("Male", 1): None,
        ("Male", 0): None,
    },
    "compas": {
        ("African-American", 1): None,
        ("African-American", 0): None,
        ("Caucasian", 1): None,
        ("Caucasian", 0): None,
        ("Other", 1): None,
        ("Other", 0): None,
    },
}

In [ ]:
# @title Check your weights (the Step 4 check, for every group)
w_mine = YOUR_WEIGHTS[DATASET]
missing = [k for k, v in w_mine.items() if v is None]
if missing:
    print("Fill in these weights first:", missing)
else:
    overall = train["label"].mean()
    rows = []
    for g in groups:
        t = train[train[GROUP] == g]
        n1, n0 = (t["label"] == 1).sum(), (t["label"] == 0).sum()
        weighted = n1 * w_mine[(g, 1)] / (n1 * w_mine[(g, 1)] + n0 * w_mine[(g, 0)])
        rows.append({"group": g, "weighted rate of label = 1": weighted, "overall rate": overall,
                     "matches?": "yes" if abs(weighted - overall) < 0.005 else "NO - recheck"})
    show(pd.DataFrame(rows).set_index("group"))

### 4.3 · Retrain the model with your weights
Same model as before (a logistic regression that never sees the group column), trained on the train rows, but each row now counts by its weight.

In [ ]:
# @title Build the model (just run this)
never = ["label", "split", "score", "predicted", "predicted_reweighed", "predicted_threshold",
         cfg["target"], GROUP] + cfg["drop"] + ([cfg["second"]] if cfg["second"] else [])
FEATURES = [c for c in df.columns if c not in never]
CATEGORICAL = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

def new_model():
    return make_pipeline(
        make_column_transformer((OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
                                (StandardScaler(), NUMERIC)),
        LogisticRegression(max_iter=1000))

X_train, y_train = train[FEATURES], train["label"]
X_test, y_test = test[FEATURES], test["label"]
print(f"The model uses {len(FEATURES)} columns:", ", ".join(FEATURES))

In [ ]:
# @title Retrain with your weights and measure (test rows)
if any(v is None for v in YOUR_WEIGHTS[DATASET].values()):
    print("Fill in your weights in 4.2 first.")
else:
    w = [YOUR_WEIGHTS[DATASET][(g, y)] for g, y in zip(train[GROUP], train["label"])]
    model_rw = new_model()
    model_rw.fit(X_train, y_train, logisticregression__sample_weight=w)
    pred_rw = model_rw.predict(X_test)

    mf_rw = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=pred_rw, sensitive_features=test[GROUP])
    print("Your reweighed model, by group:")
    show(mf_rw.by_group)
    print("Gap:"); show(gap(mf_rw).drop("n").to_frame("gap").T)
    agree = (pred_rw == test["predicted_reweighed"].values).mean()
    print(f"\nYour model agrees with the file's predicted_reweighed column on {agree:.1%} of test rows.")

## Section 5 · Thresholds  *(Session 2, Exercise 5)*
### 5.1 · The threshold game
The model's scores stay the same; you choose a cutoff for each group. The **fixed** group keeps one slider; every other group uses the second slider.
Move a slider, then press ▶ on the cell. Round 1: make the selection rates match. Round 2: make the TPRs match.

In [ ]:
# @title Threshold game
CUTOFF_FIXED_GROUP = 0.5  # @param {type:"slider", min:0.01, max:0.99, step:0.01}
CUTOFF_OTHER_GROUPS = 0.5  # @param {type:"slider", min:0.01, max:0.99, step:0.01}

cut = np.where(test[GROUP] == cfg["fixed"], CUTOFF_FIXED_GROUP, CUTOFF_OTHER_GROUPS)
pred_game = (test["score"] >= cut).astype(int)
mf_game = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=pred_game, sensitive_features=test[GROUP])
print(f"Fixed group ({cfg['fixed']}) cutoff: {CUTOFF_FIXED_GROUP:.2f}    other groups' cutoff: {CUTOFF_OTHER_GROUPS:.2f}")
show(mf_game.by_group[["selection rate", "TPR", "FPR", "accuracy"]])
print("Gap:"); show(gap(mf_game)[["selection rate", "TPR", "FPR"]].to_frame("gap").T)

### 5.2 · Fairlearn's `ThresholdOptimizer`
It plays the game for you: for each group it tries every cutoff and keeps the combination with the best accuracy that meets a fairness **constraint**.
The file's `predicted_threshold` column used **true_positive_rate_parity** (equal TPR). Try the others too.

Fairlearn's own warnings: it needs each person's group when it makes a prediction, it often costs accuracy, and separate cutoffs by group may count as *disparate treatment* under US anti-discrimination law.

In [ ]:
# @title ThresholdOptimizer
CONSTRAINT = "true_positive_rate_parity"  # @param ["true_positive_rate_parity", "demographic_parity", "false_positive_rate_parity", "equalized_odds"]

model = new_model().fit(X_train, y_train)          # the original model
to = ThresholdOptimizer(
    estimator=model,
    constraints=CONSTRAINT,
    objective="accuracy_score",
    prefit=True,
    predict_method="predict_proba",
)
to.fit(X_train, y_train, sensitive_features=train[GROUP])
# predict every row with the same random seed as the class files, then keep the test rows
pred_all = to.predict(df[FEATURES], sensitive_features=df[GROUP], random_state=42)
pred_to = np.asarray(pred_all)[(df["split"] == "test").values]

mf_to = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=pred_to, sensitive_features=test[GROUP])
print(f"Constraint: {CONSTRAINT}   (test rows)")
show(mf_to.by_group)
print("Gap:"); show(gap(mf_to).drop("n").to_frame("gap").T)
if CONSTRAINT == "true_positive_rate_parity":
    agree = (np.asarray(pred_to) == test["predicted_threshold"].values).mean()
    print(f"\nAgrees with the file's predicted_threshold column on {agree:.1%} of test rows (should be at or near 100%).")
print("\nThe constraint is met on the TRAIN rows. On new (test) rows it holds only roughly.")

## Section 6 · Compare the three models  *(Session 2, Exercise 6)*
These use the three prediction columns in your file (`predicted`, `predicted_reweighed`, `predicted_threshold`), so the numbers match your spreadsheet and the handout exactly.

In [ ]:
# @title Before and after (test rows)
MODELS = {"original": "predicted", "reweighed": "predicted_reweighed", "threshold": "predicted_threshold"}
by_group, gaps = [], {}
for name, col in MODELS.items():
    m = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=test[col], sensitive_features=test[GROUP])
    bg = m.by_group.copy(); bg.insert(0, "model", name)
    by_group.append(bg.reset_index())
    gaps[name] = gap(m)[["selection rate", "TPR", "FPR", "accuracy"]]
print("By group:")
show(pd.concat(by_group).set_index(["model", GROUP]))
gap_table = pd.DataFrame(gaps).T
print("\nGap (larger - smaller of the two compared groups) for each model:")
show(gap_table)

In [ ]:
# @title Chart: gaps before and after
ax = gap_table[["selection rate", "TPR", "FPR"]].plot.bar(color=COLORS, figsize=(8, 4), rot=0, width=0.75)
ax.set_ylabel("gap between groups"); ax.set_title(f"{DATASET}: gaps by model (test rows, smaller = more equal)")
ax.grid(axis="y", alpha=0.3); plt.show()

### Take these questions to your memo
- Which gap did each method shrink? Did any gap **grow**?
- How much accuracy did each method cost, overall and for each group?
- In your dataset, which error matters most, and for whom? Which model would you recommend?
- What would no metric fix here (a biased label, small groups, whether this decision should be automated at all)?

*Datasets: Adult (Becker & Kohavi, 1996), UCI Machine Learning Repository, CC BY 4.0; COMPAS data from ProPublica (Angwin et al., 2016). Library: Fairlearn (Weerts et al., 2023).*